In [45]:
# %matplotlib qt
import py4DSTEM
import hyperspy.api as hs
import numpy as np
import matplotlib.pyplot as plt
import glob
import json, os, gc
from py4DSTEM.visualize import show


import logging

logger = logging.getLogger('myLogger')
logger.setLevel(logging.DEBUG)
# logger.addHandler(console_handler())

logger.debug(f"py4DSTEM.__file__ is {py4DSTEM.__file__}")
logger.debug(f"py4DSTEM.__version__ is {py4DSTEM.__version__}")
from scipy.ndimage import gaussian_filter

DEBUG: py4DSTEM.__file__ is /home/eha56862/.local/lib/python3.10/site-packages/py4DSTEM/__init__.py
DEBUG: py4DSTEM.__version__ is 0.14.19


In [ ]:
# Leave empty!

In [4]:
_base_path = os.path.dirname(raw_data_path)
_mask_path = os.path.join(_base_path, 'mask.npy')
mask = np.load(_mask_path)
_labels_path = os.path.join(_base_path, 'labels.npy')
labels = np.load(_labels_path)

In [5]:
# Read cal info
cal_path = '/dls/e02/data/2026/cm44133-1/processing/Merlin/au_xgrating_150k/20260225_091202/20260225_091202_CL_40cm.json'
with open(cal_path, 'r') as f:
    cal_params = json.load(f)
print(cal_params)

pixel_size_inv_Ang = cal_params['reciprocal_space_pix(1/A)']
probe_step_size_Ang = 52.08  
p_ellipse = (cal_params['p_ellipse'])


{'reciprocal_space_pix(1/A)': 0.00565428128906469, 'p_ellipse': [301.68628058022045, 319.4845145484991, 127.7957669634648, 122.91694964604642, -2.892431305099378], 'nominal_camera_length(m)': 0.4}


In [6]:
root_path = os.path.dirname(raw_data_path)
print(root_path) 
logger.debug(f'root path is: {root_path}')
save_path = os.path.join(root_path, save_path_name)
logger.debug(f'saving path is: {save_path}')
if not os.path.exists(save_path):
    os.mkdir(save_path)

time_stamp = root_path.split('/')[-1]
print(time_stamp)

DEBUG: root path is: /dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_163652
DEBUG: saving path is: /dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_163652/ACOM


/dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_163652
20260224_163652


In [7]:
if load_prepared_data == '0':
    d = hs.load(raw_data_path, reader='HSPY')
    logger.debug(f'raw dataset shape after loading is: {d.data.shape}')
    
    if fill_cross == '1':
        d_cross_rm = map(epsic.warp_3d.remove_cross, d.data)
        d_cross_rm = list(d_cross_rm)
        d = hs.signals.Signal2D(d_cross_rm)
        d_mean = d.mean()
    else:
        d_mean = d.mean()
    
    v_min = float(v_min) # 0.01
    v_max = float(v_max) # 0.99
    probe_semiangle, qx0, qy0 = py4DSTEM.process.calibration.origin.get_probe_size(d_mean.data, v_min,v_max)
    
    if crop_q != '':
        crop_q = int(crop_q)
        d = d.isig[int(qx0 - crop_q):int(qx0 + crop_q), int(qy0 - crop_q):int(qy0 + crop_q)]
        logger.debug(f'cropped data to: {d.data.shape}')
    if len(d.data.shape) == 3:
        
        s1 = int(np.floor(d.data.shape[0]**0.5))
        s2 = d.data.shape[0]//s1
        s3, s4 = d.data.shape[1], d.data.shape[2]
        d = hs.signals.Signal2D(d.data.reshape((s1,s2,s3,s4)))
        d = d.inav[:-1,:]

    # Save data
    prepared_data_path = os.path.join(save_path, f'{time_stamp}_prepared_data.h5')
    py4DSTEM.io.save(prepared_data_path, d.data, mode = 'o')

Non-empty compiler output encountered. Set the environment variable PYOPENCL_COMPILER_OUTPUT=1 to see more.
Non-empty compiler output encountered. Set the environment variable PYOPENCL_COMPILER_OUTPUT=1 to see more.
DEBUG: raw dataset shape after loading is: (255, 255, 515, 515)


In [8]:
del d
gc.collect()

951

In [9]:
dataset = py4DSTEM.read(prepared_data_path)

In [ ]:
os.remove(prepared_data_path) # to open up space

In [10]:
dataset = py4DSTEM.DataCube(dataset.data)

In [11]:
# Diffraction space
dataset.calibration.set_Q_pixel_size(pixel_size_inv_Ang)
dataset.calibration.set_Q_pixel_units('A^-1')

# Real space
dataset.calibration.set_R_pixel_size(probe_step_size_Ang)
dataset.calibration.set_R_pixel_units('A')

In [12]:
hot_pix_thresh = float(hot_pix_thresh)
dataset, mask = dataset.filter_hot_pixels(thresh = hot_pix_thresh, return_mask=True)

Cleaning pixels: 100%|████████████| 65025/65025 [00:01<00:00, 62616.17 images/s]


In [13]:
plt.figure()
plt.imshow(mask)

In [14]:
dataset.get_dp_max();
dataset.get_dp_mean();

In [15]:
fig, ax = py4DSTEM.show([
        dataset.tree('dp_mean'), 
        dataset.tree('dp_max'), 
    ],
    cmap='inferno',
    power = 0.5,
    returnfig=True,
)
fig.savefig(os.path.join(save_path,'dp_mean_max.PNG'))

In [16]:
probe_semiangle, qx0, qy0

(6.672751341838528, 288.8235692305718, 312.04107202640057)

In [17]:
# Create an annular dark field (ADF) virtual detector using user-chosen values:
center = (qx0,qy0)
radii = (35,155)

# Plot the ADF detector
py4DSTEM.visualize.show(
    dataset.tree('dp_max'), 
    scaling='log',
    annulus = {
      'center':center,
      'radii':radii,
      'alpha':0.3,
      'fill':True
    }
)

# Calculate the ADF image
dataset.get_virtual_image(
    mode = 'annulus',
    geometry = ((center),radii),
    name = 'dark_field',
)

# Plot the ADF image
py4DSTEM.visualize.show(
    dataset.tree('dark_field'),
)

FigureCanvasAgg is non-interactive, and thus cannot be shown
FigureCanvasAgg is non-interactive, and thus cannot be shown
FigureCanvasAgg is non-interactive, and thus cannot be shown
100%|██████████████████████████████████| 65025/65025 [00:05<00:00, 11517.48it/s]
FigureCanvasAgg is non-interactive, and thus cannot be shown
FigureCanvasAgg is non-interactive, and thus cannot be shown
FigureCanvasAgg is non-interactive, and thus cannot be shown
FigureCanvasAgg is non-interactive, and thus cannot be shown


In [18]:
#pick random positions with high intensity from df image
n_pos = 6 # number of positinos
df_mean =dataset.tree('dark_field').data.mean()
pos = np.where(dataset.tree('dark_field').data > df_mean)
xy_pos = np.zeros(shape = (2, n_pos))
for i in range(n_pos):
    rand_int = np.random.randint(0, pos[0].shape[0])
    xy_pos[0,i] = pos[0][rand_int]
    xy_pos[1,i] = pos[1][rand_int]

In [19]:
# Choose some diffraction patterns to use for hyperparameter tuning

rxs = tuple(xy_pos[0].astype(int))#10,18,40,35,50,30
rys = tuple(xy_pos[1].astype(int)) #40,10,10,20,30,50
colors=['r','g','w','c','b','y']

py4DSTEM.visualize.show_points(
    dataset.tree('dark_field'),
    x=xy_pos[0],
    y=xy_pos[1],
    pointcolor=colors,
    figsize=(8,8)
)

In [20]:
# Try making a synthetic probe

syn_probe_rad = int(syn_probe_rad)
syn_probe_width = float(syn_probe_width)

syn_probe = py4DSTEM.braggvectors.probe.Probe.generate_synthetic_probe(syn_probe_rad, syn_probe_width, (dataset.data.shape[-1], dataset.data.shape[-1]))

In [21]:
# Construct a probe template to use as a kernel for correlation disk detection
probe_semiangle = syn_probe_rad
syn_probe_kernel = syn_probe.get_kernel(
    mode = 'sigmoid',
    radii = (probe_semiangle * 0.5, probe_semiangle * 4.0),
    bilinear=True,
)

# Plot the probe kernel
fig, ax = py4DSTEM.visualize.show_kernel(
    syn_probe.kernel, 
    R=20, 
    L=20, 
    W=1,
    figsize = (8,4),
    returnfig=True,
)
fig.savefig(os.path.join(save_path,'probe_kernel.PNG'))



In [22]:
# Test hyperparameters on a few probe positions
# Visualize the diffraction patterns and the located disk positions

# Hyperparameters
detect_params = {
    'corrPower': 1.0, #1.0,
    'sigma': 0,
    'edgeBoundary': 2,
    'minRelativeIntensity': 0.0,
    'minAbsoluteIntensity': 0.15, #0.5,
    'minPeakSpacing': 2,
    'subpixel' : 'poly',
#     'subpixel' : 'multicorr',
    'upsample_factor': 8,
    'maxNumPeaks': 1000,
    'CUDA': True,
}

disks_selected = dataset.find_Bragg_disks(
    data = (rxs, rys),
    # template = probe_kernel,
    template=syn_probe_kernel,
    **detect_params,
)


fig, ax = py4DSTEM.visualize.show_image_grid(
    get_ar = lambda i:dataset.data[rxs[i],rys[i],:,:],
    H=2, 
    W=3,
    axsize=(4,4),
    get_bordercolor = lambda i:colors[i],
    get_x = lambda i: disks_selected[i].data['qx'],
    get_y = lambda i: disks_selected[i].data['qy'],
    get_pointcolors = lambda i: colors[i],
    open_circles = True,
    scale = 100,
    intensity_range = 'absolute',
    vmin = .1,
    vmax = 2,
    returnfig=True,
)

fig.savefig(os.path.join(save_path,'peak_finding_test.PNG'))

In [23]:
bragg_peaks = dataset.find_Bragg_disks(
    template = syn_probe_kernel,
    **detect_params,
)

Using 28 batches of 2342 patterns each...


Finding Bragg disks in batches: 100%|████████| 28/28 [00:36<00:00,  1.29s/batch]


Analyzed 65025 diffraction patterns in 0.0h 0.0m 36.17s
(avg. speed 1797.9604 patterns per second)


In [24]:
# compute
bvm = bragg_peaks.histogram( mode='raw' )

# show
show(bvm,
    scaling='power',
    power=0.002)

FigureCanvasAgg is non-interactive, and thus cannot be shown
FigureCanvasAgg is non-interactive, and thus cannot be shown
FigureCanvasAgg is non-interactive, and thus cannot be shown
FigureCanvasAgg is non-interactive, and thus cannot be shown
FigureCanvasAgg is non-interactive, and thus cannot be shown
FigureCanvasAgg is non-interactive, and thus cannot be shown
FigureCanvasAgg is non-interactive, and thus cannot be shown
FigureCanvasAgg is non-interactive, and thus cannot be shown


In [25]:
bragg_peaks.setcal()

In [26]:
bragg_peaks.calstate

{'center': False, 'ellipse': False, 'pixel': True, 'rotate': False}

In [27]:
bragg_vector_map = bragg_peaks.get_bvm(mode='raw')
# bragg_vector_map_masked = bragg_peaks_masked.get_bvm(mode='raw')
# Plot a comparison image between the original and masked bragg vector map
fig, ax = py4DSTEM.show(
    [
        bragg_vector_map,
        # bragg_vector_map_masked,
    ],
    combine_images = True,
    scaling='power',
    power=0.002,
    figsize = (4,4),
    returnfig=True,
)
fig.savefig(os.path.join(save_path,'bragg_peaks.PNG'))

In [28]:
# Measure the origin
center_guess = (qx0,qy0)
# radial_range = (8,200)
qx0_meas,qy0_meas,mask_meas = bragg_peaks.measure_origin(
    center_guess=center_guess,
    score_method='distance',
    # findcenter='max',
)

fig, ax = show(
    [qx0_meas,qy0_meas],
    cmap = 'RdBu',
    # mask = mask_meas,
    returnfig=True,
)
fig.savefig(os.path.join(save_path,'bragg_peaks_measure_origin.PNG'))

In [29]:
# Fit a plane to the origins

qx0_fit,qy0_fit,qx0_residuals,qy0_residuals = bragg_peaks.fit_origin(
    # robust= True,
    # robust_steps = 10
    # robust_thresh= 1.2,
)

In [30]:
fig = plt.gcf()
fig.savefig(os.path.join(save_path, 'BF_disc_align.png'))


In [31]:
bragg_peaks.calstate

{'center': True, 'ellipse': False, 'pixel': True, 'rotate': False}

In [32]:
# Now that we've calibrated the center positions, we can re-compute
# the Bragg vector map, this time with the center correction applied

sampling = 1

# compute
bvm = bragg_peaks.histogram(
    #mode='cal',             # 'cal' is the default mode, so this line can be included or left out
    sampling = sampling,
)

# show
# overlay a circle around the center for visualization purposes
fig, ax = show(
    bvm,
    circle={
        'center' : bvm.origin,   # the centered BVM knows where its origin is 
        'R' : 4*sampling,
        'fill' : False,
        'linewidth' : 1
    },
    returnfig = True,
    scaling='power',
    power=0.002,
    #vmax=0.9
)
fig.savefig(os.path.join(save_path, 'BF_disc_align_bvm.png'))

In [33]:
# Compare this to the uncalibrated BVM - much better!

# compute raw vs. centered
bvm_r = bragg_peaks.histogram( mode='raw', sampling=sampling )
bvm_c = bragg_peaks.histogram( mode='cal', sampling=sampling )

# show
show( [bvm_r, bvm_c] ,vmax=0.99)

# show, zooming in on origin
L = 20
x,y = bvm_c.origin
import numpy as np
x0,xf = np.round([x-L,x+L]).astype(int)
y0,yf = np.round([y-L,y+L]).astype(int)

show(
    [
    bvm_r[x0:xf,y0:yf],
    bvm_c[x0:xf,y0:yf]
    ],
    vmax=0.9
)

In [34]:
fig = plt.gcf()
fig.savefig(os.path.join(save_path, 'BF_disc_align_before_after.png'))

In [35]:
bragg_peaks.calibration.set_p_ellipse(p_ellipse)

In [36]:
bragg_peaks.calibration.set_QR_rotflip((0, False))

In [37]:
# Save calibrated Bragg peaks
filepath_braggdisks_cali = os.path.join(save_path, f'{time_stamp}_braggdisks_cali.h5')
py4DSTEM.save(
    filepath_braggdisks_cali,
    bragg_peaks,
    mode='o',
)

100%|██████████████████████████████████| 65025/65025 [00:03<00:00, 17470.37it/s]


In [38]:


# Define fcc Pt structure using manual input of the crystal structure
pos = [
    [0.0, 0.0, 0.0],
    [0.0, 0.5, 0.5],
    [0.5, 0.0, 0.5],
    [0.5, 0.5, 0.0],
]
atom_num = 78
a = 3.94
cell = a

crystal = py4DSTEM.process.diffraction.Crystal(
    pos, 
    atom_num, 
    cell)

k_max = 0.9
crystal.calculate_structure_factors(k_max)

In [39]:
crystal.plot_scattering_intensity(
    bragg_peaks = bragg_peaks,
    bragg_k_power = 2.0,
)

In [40]:
plt.close('all')

In [41]:
acom_params = {
    'zone_axis_range': 'auto',
    'angle_step_zone_axis': 1.0,
    'angle_step_in_plane': 1.0,
    'accel_voltage': 300e3,
    'corr_kernel_size': 0.08,
#     'tol_peak_delete': 0.04,
    # 'intensity_power': 0.125,
#     'intensity_power': 0.0,
    'CUDA': True,
}
crystal.orientation_plan(**acom_params)
sigma_compare = 0.02

Automatically detected point group m-3m,
 using arguments: zone_axis_range = 
[[0 1 1]
 [1 1 1]], 
 fiber_axis=None, fiber_angles=None.


Orientation plan: 100%|████████████| 406/406 [00:00<00:00, 39108.18 zone axes/s]


In [53]:
plt.figure()
plt.imshow(labels)

In [51]:
label = 1
num = 1
coords = np.argwhere(labels == label)
test_coord = coords[np.random.choice(coords.shape[0], num, replace = False)]

In [60]:

# plotting parameters
plot_params = {
    'scale_markers': 2000,
    'scale_markers_compare': 40,
    'plot_range_kx_ky': crystal.k_max,
    'min_marker_size': 2,
    'add_labels': False,
}



# Find best fit orientations
orientation  = crystal.match_single_pattern(
    bragg_peaks.cal[test_coord[0][0],test_coord[0][1]],
    num_matches_return = 1,
    verbose = True,
)


colors=['r']

py4DSTEM.visualize.show_points(
    dataset.tree('dark_field'),
    x=test_coord[0][0],
    y=test_coord[0][1],
    pointcolor=colors,
    figsize=(8,8)
)

Best fit lattice directions: z axis = ([0.514 0.514 0.687]), x axis = ([0.096 0.549 0.83 ]), with corr value = 1.662


In [62]:
# Simulated bragg peaks from best fit orientations
peaks_fit = crystal.generate_diffraction_pattern(
    orientation,
    sigma_excitation_error=sigma_compare)

fig,ax = plt.subplots(1,2,figsize=(8,8))

py4DSTEM.process.diffraction.plot_diffraction_pattern(
    peaks_fit,
    bragg_peaks_compare=bragg_peaks.cal[test_coord[0][0],test_coord[0][1]],
    **plot_params,
    input_fig_handle=(fig,[ax[0]]),
)


In [63]:
# Fit orientation to all probe positions
orientation_map = crystal.match_orientations(
    bragg_peaks,
    num_matches_return = 1,
    min_number_peaks = 3,
)

Matching Orientations: 100%|███| 65025/65025 [00:04<00:00, 13638.82 PointList/s]


In [68]:
# plot orientation map
images_orientation, fig, ax = crystal.plot_orientation_maps(
    orientation_map,
    corr_range = np.array([1,3 ]),
    corr_normalize = False,
    camera_dist = 10,
    show_axes = False,
    figsize = (12,3),
    returnfig = True,
)

FigureCanvasAgg is non-interactive, and thus cannot be shown


In [ ]:
fig.savefig(os.path.join(save_path, 'orientation_map.png'))

In [67]:
crystal.save_ang_file(
    file_name=os.path.join(save_path,'orientation_map_crystal.ang'),
    orientation_map = orientation_map)